# Data Preparation Validation

Step-by-step validation of transformations in `src/data_preparation.py`

In [ ]:
import pandas as pd
from pathlib import Path

data_dir = Path('../data')

## 1. Geolocation - Multiple points per zip code?

Check if `get_geolocation_lookup` averaging is necessary.

In [ ]:
geo = pd.read_csv(data_dir / 'olist_geolocation_dataset.csv')
geo_counts = geo.groupby('geolocation_zip_code_prefix').size()

print(f'Total rows: {len(geo):,}')
print(f'Unique zip codes: {geo_counts.shape[0]:,}')
print(f'Zip codes with >1 point: {(geo_counts > 1).sum():,}')
print(f'Max points per zip: {geo_counts.max()}')
print(f'Mean points per zip: {geo_counts.mean():.1f}')

# Conclusion: Is averaging necessary?
print(f'\n=> Averaging IS necessary: {(geo_counts > 1).any()}')

Total rows: 1,000,163
Unique zip codes: 19,015
Zip codes with >1 point: 17,972
Max points per zip: 1146
Mean points per zip: 52.6

=> Averaging IS necessary: True


## 2. Order Items - Multiple items per order?

Check if `sum(freight_value)` makes sense.

In [ ]:
items = pd.read_csv(data_dir / 'olist_order_items_dataset.csv')
items_per_order = items.groupby('order_id').size()

print(f'Total rows: {len(items):,}')
print(f'Unique orders: {items_per_order.shape[0]:,}')
print(f'Orders with >1 item: {(items_per_order > 1).sum():,} ({(items_per_order > 1).mean():.1%})')
print(f'Max items per order: {items_per_order.max()}')

# Distribution
print('\nItems per order distribution:')
print(items_per_order.value_counts().sort_index().head(10))

Total rows: 112,650
Unique orders: 98,666
Orders with >1 item: 9,803 (9.9%)
Max items per order: 21

Items per order distribution:
1     88863
2      7516
3      1322
4       505
5       204
6       198
7        22
8         8
9         3
10        8
Name: count, dtype: int64


## 3. Sellers - Multiple sellers per order?

Check if `first(seller_id)` loses information.

In [ ]:
sellers_per_order = items.groupby('order_id')['seller_id'].nunique()

print(f'Orders with >1 seller: {(sellers_per_order > 1).sum():,} ({(sellers_per_order > 1).mean():.1%})')
print(f'Max sellers per order: {sellers_per_order.max()}')

# Show example orders with multiple sellers
if (sellers_per_order > 1).any():
    multi_seller_orders = sellers_per_order[sellers_per_order > 1].index[:3]
    print('\nExample orders with multiple sellers:')
    display(items[items['order_id'].isin(multi_seller_orders)][['order_id', 'seller_id', 'freight_value']])

Orders with >1 seller: 1,278 (1.3%)
Max sellers per order: 5

Example orders with multiple sellers:


,order_id,seller_id,freight_value
80,002f98c0f7efd42638ed6100ca699b42,7299e27ed73d2ad986de7f7c77d919fa,32.57
81,002f98c0f7efd42638ed6100ca699b42,fa40cc5b934574b62717c68f3d678b6d,7.16
296,00bcee890eba57a9767c7b5ca12d3a1b,3bb548e3cb7f70f28e3f11ee9dce0e59,15.80
297,00bcee890eba57a9767c7b5ca12d3a1b,9c0e69c7bf2619675bbadf47b43f655a,52.69
298,00bcee890eba57a9767c7b5ca12d3a1b,3bb548e3cb7f70f28e3f11ee9dce0e59,15.80
299,00bcee890eba57a9767c7b5ca12d3a1b,3bb548e3cb7f70f28e3f11ee9dce0e59,15.80
465,01144cadcf64b6427f0a6580a3033220,620c87c171fb2a6dd6e8bb4dec959fc6,24.70
466,01144cadcf64b6427f0a6580a3033220,abcd2cb37d46c2c8fb1bf071c859fc5b,24.70


## 4. Payments - Multiple payments per order?

Check payment aggregation logic.

In [ ]:
payments = pd.read_csv(data_dir / 'olist_order_payments_dataset.csv')
payments_per_order = payments.groupby('order_id').size()

print(f'Total rows: {len(payments):,}')
print(f'Unique orders: {payments_per_order.shape[0]:,}')
print(f'Orders with >1 payment: {(payments_per_order > 1).sum():,} ({(payments_per_order > 1).mean():.1%})')
print(f'Max payments per order: {payments_per_order.max()}')

Total rows: 103,886
Unique orders: 99,440
Orders with >1 payment: 2,961 (3.0%)
Max payments per order: 29


In [ ]:
# What payment types are used in multi-payment orders?
multi_payment_orders = payments_per_order[payments_per_order > 1].index
multi_payments = payments[payments['order_id'].isin(multi_payment_orders)]

print('Payment type combinations in multi-payment orders:')
type_combos = multi_payments.groupby('order_id')['payment_type'].apply(lambda x: tuple(sorted(x.unique())))
print(type_combos.value_counts().head(10))

print('\nExample multi-payment order:')
display(payments[payments['order_id'] == multi_payment_orders[0]])

Payment type combinations in multi-payment orders:
payment_type
(credit_card, voucher)       2245
(voucher,)                    427
(credit_card,)                287
(credit_card, debit_card)       1
(debit_card,)                   1
Name: count, dtype: int64

Example multi-payment order:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
80856,0016dfedd97fc2950e388d2971d718c7,2,voucher,1,17.92
89575,0016dfedd97fc2950e388d2971d718c7,1,credit_card,5,52.63


## 5. Products - Multiple categories per order?

Check if `first(product_category)` loses information.

In [ ]:
products = pd.read_csv(data_dir / 'olist_products_dataset.csv')
items_products = items.merge(products, on='product_id', how='left')

cats_per_order = items_products.groupby('order_id')['product_category_name'].nunique()

print(f'Orders with >1 category: {(cats_per_order > 1).sum():,} ({(cats_per_order > 1).mean():.1%})')
print(f'Max categories per order: {cats_per_order.max()}')

# How many unique categories exist?
print(f'\nTotal unique categories: {products["product_category_name"].nunique()}')
print(f'Null categories: {products["product_category_name"].isna().sum()}')

Orders with >1 category: 727 (0.7%)
Max categories per order: 3

Total unique categories: 73
Null categories: 610
